In [2]:
!pip install dbrepo python-dotenv

In [3]:
from dbrepo.RestClient import RestClient
from dotenv import load_dotenv
import os

load_dotenv()
client = RestClient(
    "https://test.dbrepo.tuwien.ac.at/",
    username=os.getenv("DBREPO_USER"),
    password=os.getenv("DBREPO_PASS")
)

DATABASE_ID = "cf27a11d-58e5-4693-856c-e8f3527e3394"

tables = client.get_tables(DATABASE_ID)
for t in tables:
    print(f"Table: {t.name}, ID: {t.id}")
    table_detail = client.get_table(DATABASE_ID, t.id)
    for col in table_detail.columns:
        print(f"  Column: {col.name}, ID: {col.id}")

Table: wastewater_data, ID: aa6cbf8c-f35e-4411-81ae-20f1a1acf682
  Column: city_name, ID: cfd5f149-5c73-461c-9b2e-b45c370f6203
  Column: ref_year, ID: d61d3210-3a81-4d4e-9d06-f7a024523779
  Column: metabolite_name, ID: 798011fc-487b-4dcc-b640-fb3ec4e374dd
  Column: daily_mean, ID: 4dbd8624-4a0b-4a2d-a7e9-e8b745600c6f
Table: gdp_data, ID: 0016c161-ead4-49ce-b8ea-a48c2797eeab
  Column: nuts_code, ID: 85b5a0d7-845c-4a66-9ffb-20d2147f6d76
  Column: ref_year, ID: 9741a260-9bd8-488a-b6cd-bca545086614
  Column: gdp_per_cap, ID: 02d008c9-1128-46ac-94e8-4560a2d98301
Table: city_map, ID: 6c1a3235-df5c-4bc8-bbd0-b7dce088fcfe
  Column: nuts_code, ID: 70571097-af37-4cd3-9916-270cb3d4548b
  Column: city_name, ID: e0ddcf5f-c7b7-438e-be85-6c27c94d4a69


In [16]:
mappings = [
    {
        "table_name": "city_map",
        "column": "nuts_code",
        "concept_uri": "http://purl.org/linked-data/sdmx/2009/dimension#refArea"
    },
    {
        "table_name": "city_map",
        "column": "city_name",
        "concept_uri": "http://purl.obolibrary.org/obo/NCIT_C95378"
    },
    {
        "table_name": "gdp_data",
        "column": "ref_year",
        "concept_uri": "http://rs.tdwg.org/dwc/terms/year"
    },
    {
        "table_name": "gdp_data",
        "column": "gdp_per_cap",
        "concept_uri": "http://purl.org/linked-data/sdmx/2009/measure#obsValue"
    },
    {
        "table_name": "wastewater_data",
        "column": "metabolite_name",
        "concept_uri": "http://purl.obolibrary.org/obo/CHEBI_23367"
    },
    {
        "table_name": "wastewater_data",
        "column": "daily_mean",
        "concept_uri": "http://purl.allotrope.org/ontologies/process#AFP_0002800"
    }
]

import requests
from requests.auth import HTTPBasicAuth
import os

BASE_URL = "https://test.dbrepo.tuwien.ac.at"

for m in mappings:

    print("\n----------------------")
    print(f"{m['table_name']}.{m['column']}")

    table = next((t for t in tables if t.name == m["table_name"]), None)

    if not table:
        print("Table not found")
        continue

    table_detail = client.get_table(DATABASE_ID, table.id)

    col = next((c for c in table_detail.columns if c.name == m["column"]), None)

    if not col:
        print("Column not found")
        continue

    print("Table ID:", table.id)
    print("Column ID:", col.id)

    url = f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{table.id}/column/{col.id}"

    payload = {
        "concept_uri": m["concept_uri"]
    }

    response = requests.put(
        url,
        json=payload,
        auth=HTTPBasicAuth(
            os.getenv("DBREPO_USER"),
            os.getenv("DBREPO_PASS")
        )
    )

    print("STATUS:", response.status_code)

    if response.ok:
        print("Success")
    else:
        print("Failed")
        print(response.text)


----------------------
city_map.nuts_code
Table ID: 6c1a3235-df5c-4bc8-bbd0-b7dce088fcfe
Column ID: 70571097-af37-4cd3-9916-270cb3d4548b
STATUS: 202
Success

----------------------
city_map.city_name
Table ID: 6c1a3235-df5c-4bc8-bbd0-b7dce088fcfe
Column ID: e0ddcf5f-c7b7-438e-be85-6c27c94d4a69
STATUS: 202
Success

----------------------
gdp_data.ref_year
Table ID: 0016c161-ead4-49ce-b8ea-a48c2797eeab
Column ID: 9741a260-9bd8-488a-b6cd-bca545086614
STATUS: 202
Success

----------------------
gdp_data.gdp_per_cap
Table ID: 0016c161-ead4-49ce-b8ea-a48c2797eeab
Column ID: 02d008c9-1128-46ac-94e8-4560a2d98301
STATUS: 202
Success

----------------------
wastewater_data.metabolite_name
Table ID: aa6cbf8c-f35e-4411-81ae-20f1a1acf682
Column ID: 798011fc-487b-4dcc-b640-fb3ec4e374dd
STATUS: 202
Success

----------------------
wastewater_data.daily_mean
Table ID: aa6cbf8c-f35e-4411-81ae-20f1a1acf682
Column ID: 4dbd8624-4a0b-4a2d-a7e9-e8b745600c6f
STATUS: 202
Success


## Brief explanation of used ontologies
The semantic mappings were implemented using statistical and biomedical ontologies, primarily SDMX, ChEBI, and NCIt. SDMX was selected for statistical and regional indicators because it is widely used by organizations such as Eurostat and OECD, while ChEBI and Allotrope ontologies were used for domain-specific chemical and analytical concepts related to wastewater epidemiology data.